# Take the frame based detections and turn them into movement trajectories
(For speed, uses multiprocessing and breaks up observations into 10 overlapping temporal groups to process seperately since tracks are much shorter in time than the total length of the obervations.)

In [2]:
import glob
import os

import matplotlib.pyplot as plt
from multiprocessing import Pool
import numpy as np

import bat_functions as kbf

In [3]:
base_folder = r"H:/kasanka-bats/"
# Which day to track
day = "20221201"
#"20211201"
#"20211207"
#"20211215"
camera_folders = sorted(glob.glob(os.path.join(base_folder, day, '*')))

In [4]:
base_folder = r"H:/kasanka-bats/"
year_prefix = "2022"

# List all directories in base_folder that start with the year prefix
days = [f for f in os.listdir(base_folder) if os.path.isdir(os.path.join(base_folder, f)) and f.startswith(year_prefix)]
print(days)

['20221101', '20221116', '20221124', '20221201', '20221213', '20221219']


In [5]:
base_folder = r"H:\kasanka-bats"
# Which days to track
# days = ["20221213"]
camera_folders = []

for day in days:
    print(day)
    day_camera_folders = sorted(glob.glob(os.path.join(base_folder, day, '*')))
    camera_folders.extend(day_camera_folders)
    #day_camera_folders = sorted(glob.glob(os.path.join(base_folder, day, 'MusolaTower*')))
    #camera_folders.extend(day_camera_folders)
    #day_camera_folders = sorted(glob.glob(os.path.join(base_folder, day, 'MusolaPath2*')))
    #camera_folders.extend(day_camera_folders)
    #day_camera_folders = sorted(glob.glob(os.path.join(base_folder, day, 'Sunset*')))
    #camera_folders.extend(day_camera_folders)

20221101
20221116
20221124
20221201
20221213
20221219


In [6]:
print(camera_folders)

['H:\\kasanka-bats\\20221101\\1 Fibwe Parking', 'H:\\kasanka-bats\\20221101\\10 Fibwe Management', 'H:\\kasanka-bats\\20221101\\2 BBC', 'H:\\kasanka-bats\\20221101\\3 Chinyangali', 'H:\\kasanka-bats\\20221101\\4 Not Chinyangali', 'H:\\kasanka-bats\\20221101\\5 Puku', 'H:\\kasanka-bats\\20221101\\6 Sunset', 'H:\\kasanka-bats\\20221101\\7 Bupata', 'H:\\kasanka-bats\\20221101\\8 Musola Path', 'H:\\kasanka-bats\\20221101\\9 KK', 'H:\\kasanka-bats\\20221116\\1 Fibwe Parking', 'H:\\kasanka-bats\\20221116\\10 Fibwe Management', 'H:\\kasanka-bats\\20221116\\2 BBC', 'H:\\kasanka-bats\\20221116\\3 Chinyangali', 'H:\\kasanka-bats\\20221116\\4 Not Chinyangali', 'H:\\kasanka-bats\\20221116\\5 Puku', 'H:\\kasanka-bats\\20221116\\6 Sunset', 'H:\\kasanka-bats\\20221116\\7 Bupata', 'H:\\kasanka-bats\\20221116\\8 Musola Path', 'H:\\kasanka-bats\\20221116\\9 KK', 'H:\\kasanka-bats\\20221124\\1 Fibwe Parking', 'H:\\kasanka-bats\\20221124\\10 Fibwe Management', 'H:\\kasanka-bats\\20221124\\2 BBC', 'H:\\kas

In [7]:
n_camera_folders = []
for folder in camera_folders:
    tracks_file = os.path.join(folder, "raw_tracks.npy")
    if os.path.exists(tracks_file):
        # Skip videos that have already been tracked
        continue
    else:
        n_camera_folders.append(folder)

print("Videos to track...")
print(*n_camera_folders, sep='\n')

Videos to track...
H:\kasanka-bats\20221101\1 Fibwe Parking
H:\kasanka-bats\20221101\10 Fibwe Management
H:\kasanka-bats\20221101\2 BBC
H:\kasanka-bats\20221101\3 Chinyangali
H:\kasanka-bats\20221101\4 Not Chinyangali
H:\kasanka-bats\20221101\5 Puku
H:\kasanka-bats\20221101\6 Sunset
H:\kasanka-bats\20221101\7 Bupata
H:\kasanka-bats\20221101\8 Musola Path
H:\kasanka-bats\20221101\9 KK
H:\kasanka-bats\20221116\1 Fibwe Parking
H:\kasanka-bats\20221116\10 Fibwe Management
H:\kasanka-bats\20221116\2 BBC
H:\kasanka-bats\20221116\3 Chinyangali
H:\kasanka-bats\20221116\4 Not Chinyangali
H:\kasanka-bats\20221116\5 Puku
H:\kasanka-bats\20221116\6 Sunset
H:\kasanka-bats\20221116\7 Bupata
H:\kasanka-bats\20221116\8 Musola Path
H:\kasanka-bats\20221116\9 KK
H:\kasanka-bats\20221124\1 Fibwe Parking
H:\kasanka-bats\20221124\10 Fibwe Management
H:\kasanka-bats\20221124\2 BBC
H:\kasanka-bats\20221124\3 Chinyangali
H:\kasanka-bats\20221124\4 Not Chinyangali
H:\kasanka-bats\20221124\5 Puku
H:\kasanka-bat

In [8]:
def track(camera_dict):
    camera_folder = camera_dict['camera_folder']
    if glob.glob(os.path.join(camera_folder, 'first_frame*.npy')):
        print(f"Skipping {os.path.basename(camera_folder)} since files exist.")
        return
    first_frame = camera_dict['first_frame']
    max_frame = camera_dict['max_frame']
    print(f"{os.path.basename(camera_folder)} begun.")
    contours_files = sorted(
        glob.glob(os.path.join(camera_folder, 'contours-compressed-*.npy'))
    )
    if contours_files:
        contours_files = contours_files[1:]
        centers = np.load(os.path.join(camera_folder, 'p_centers.npy'), allow_pickle=True)
        sizes = np.load(os.path.join(camera_folder, 'p_size.npy'), allow_pickle=True)
        tracks_file = os.path.join(camera_folder, f'first_frame_{first_frame}_max_val_{max_frame}_raw_tracks.npy')
        raw_tracks = kbf.find_tracks(first_frame, centers, contours_files=contours_files, 
                                     sizes_list=sizes, tracks_file=tracks_file,
                                     max_frame=max_frame)
    else:
        print("Missing contour files.")

In [9]:
camera_dicts = []
for camera_folder in n_camera_folders:
    # To speed up processing, detections found in each observation are split
    # into 10 groups by time with 15 seconds of overlap in each group
    centers_file = os.path.join(camera_folder, 'p_centers.npy')
    
    # Skip if p_centers.npy does not exist
    if not os.path.exists(centers_file):
        print(f"No centers file found in {camera_folder}, skipping...")
        continue
        
    centers = np.load(centers_file, allow_pickle=True)
    max_vals = np.linspace(0, len(centers), 10, dtype=int)[1:].tolist()
    max_vals[-1] = None
    min_vals = np.linspace(0, len(centers), 10, dtype=int)[:-1]
    # 15 second overlap
    min_vals[1:] = min_vals[1:] - 450
    for min_val, max_val in zip(min_vals, max_vals):
        min_val = np.max([min_val, 0])
        camera_dict = {'camera_folder': camera_folder,
                       'first_frame': min_val,
                       'max_frame': max_val}
        if max_val is None:
            tracks_basename = f'first_frame_{min_val:06d}_max_val_{max_val}_raw_tracks.npy'
        else:
            tracks_basename = f'first_frame_{min_val:06d}_max_val_{max_val:06d}_raw_tracks.npy'
        tracks_file = os.path.join(camera_folder, tracks_basename)
        if not os.path.exists(tracks_file):
            print(tracks_file)
            camera_dicts.append(camera_dict)
        

No centers file found in H:\kasanka-bats\20221101\1 Fibwe Parking, skipping...
No centers file found in H:\kasanka-bats\20221101\10 Fibwe Management, skipping...
No centers file found in H:\kasanka-bats\20221101\2 BBC, skipping...
No centers file found in H:\kasanka-bats\20221101\3 Chinyangali, skipping...
No centers file found in H:\kasanka-bats\20221101\4 Not Chinyangali, skipping...
No centers file found in H:\kasanka-bats\20221101\5 Puku, skipping...
No centers file found in H:\kasanka-bats\20221101\6 Sunset, skipping...
No centers file found in H:\kasanka-bats\20221101\7 Bupata, skipping...
No centers file found in H:\kasanka-bats\20221101\8 Musola Path, skipping...
No centers file found in H:\kasanka-bats\20221101\9 KK, skipping...
No centers file found in H:\kasanka-bats\20221116\1 Fibwe Parking, skipping...
No centers file found in H:\kasanka-bats\20221116\10 Fibwe Management, skipping...
No centers file found in H:\kasanka-bats\20221116\2 BBC, skipping...
No centers file found

In [10]:
print(camera_dicts)
#with Pool(processes=5) as pool:
#    pool.map(track, camera_dicts)

[]


In [11]:

for camera_dict in camera_dicts:
    track(camera_dict)

### Now connect the 10 sections of the observation that were tracked seperately together

In [12]:
def combine_overlapping_tracks(observation_folder, first_group=0, last_group=None, save=False):
    track_files = glob.glob(os.path.join(observation_folder, 'first_frame*.npy'))
    track_files = sorted(track_files, key=lambda f: int(f.split('_')[-6]))

    track_groups = []
    for file in track_files:
        track_groups.append(np.load(file, allow_pickle=True))
        
        
    for track_file in track_files:
        print(os.path.basename(track_file))
        
    first_overlap_frames = [int(f.split('_')[-6]) for f in track_files[1:]]
    first_overlap_frames.append(None)
    print(first_overlap_frames)
        
    all_tracks = []

    total_tracks = 0
    for group_ind, track_group in enumerate(track_groups[first_group:last_group]):
        if group_ind >= len(track_groups) -1:
            for track in track_group:
                if type(track['track']) == list:
                    track['track'] = np.stack(track['track'])
                    track['pos_index'] = np.stack(track['pos_index'])
                    if 'size' in track:
                        track['size'] = np.stack(track['size'])
                all_tracks.append(track)
            total_tracks += len(track_group)
            break

        for track_ind, track in enumerate(track_group):
            if track['first_frame'] < first_overlap_frames[first_group + group_ind]:
                all_tracks.append(track)


    all_tracks_file = os.path.join(observation_folder, 'raw_tracks.npy')
    if save:
        np.save(all_tracks_file, all_tracks)
        print('saved')

In [13]:
observation_folders = []
for folder in camera_folders:
    if not os.path.exists(os.path.join(folder, 'first_frame*.npy')):
        observation_folders.append(folder)

In [14]:
print(observation_folders)

['H:\\kasanka-bats\\20221101\\1 Fibwe Parking', 'H:\\kasanka-bats\\20221101\\10 Fibwe Management', 'H:\\kasanka-bats\\20221101\\2 BBC', 'H:\\kasanka-bats\\20221101\\3 Chinyangali', 'H:\\kasanka-bats\\20221101\\4 Not Chinyangali', 'H:\\kasanka-bats\\20221101\\5 Puku', 'H:\\kasanka-bats\\20221101\\6 Sunset', 'H:\\kasanka-bats\\20221101\\7 Bupata', 'H:\\kasanka-bats\\20221101\\8 Musola Path', 'H:\\kasanka-bats\\20221101\\9 KK', 'H:\\kasanka-bats\\20221116\\1 Fibwe Parking', 'H:\\kasanka-bats\\20221116\\10 Fibwe Management', 'H:\\kasanka-bats\\20221116\\2 BBC', 'H:\\kasanka-bats\\20221116\\3 Chinyangali', 'H:\\kasanka-bats\\20221116\\4 Not Chinyangali', 'H:\\kasanka-bats\\20221116\\5 Puku', 'H:\\kasanka-bats\\20221116\\6 Sunset', 'H:\\kasanka-bats\\20221116\\7 Bupata', 'H:\\kasanka-bats\\20221116\\8 Musola Path', 'H:\\kasanka-bats\\20221116\\9 KK', 'H:\\kasanka-bats\\20221124\\1 Fibwe Parking', 'H:\\kasanka-bats\\20221124\\10 Fibwe Management', 'H:\\kasanka-bats\\20221124\\2 BBC', 'H:\\kas

In [15]:
for folder in observation_folders:
    combine_overlapping_tracks(folder, save=True)

[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
first_frame_0_max_val_7070_raw_tracks.npy
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
[None]
saved
